# Vehicle Attrition as a Signal for Emerging Vehicle Collectability

**Author:** Phillip Smith\
**Course:** Machine Learning & AI\
**Term:** Fall 2026\
**Employer Partner:** Hagerty

---

## Notebook Overview

This notebook covers the exploratory data analysis (EDA) and feature engineering phase of a semester-long capstone project investigating vehicle attrition patterns and their potential as early signals of emerging vehicle collectability.

The analysis draws on two integrated datasets — VIO (Vehicles in Operation) registration snapshots and VINtelligence vehicle specification data — unified into a single cohort-level table in Snowflake (`WORKSPACE.ENGINEER.VIO_COHORT_ENRICHED`). All data preparation and integration work was completed prior to this notebook and is documented in the project decision log.

**Scope:** cars and trucks only (16,188 cohorts), including light and medium duty commercial vehicles. Motorcycle data is prepared in Snowflake but excluded from this analysis.

---

## Business Context

Hagerty is a specialty automotive lifestyle and insurance company focused on classic and collector vehicles. As a production vehicle batch ages, its registered population declines. Year-over-year shifts in that attrition rate may serve as an early signal of emerging collectability — as supply shrinks, scarcity can pressure values upward and slow further attrition, creating a feedback loop between scarcity and desirability. Early identification of these signals could enable Hagerty to identify emerging enthusiast markets before they mature.

---

## Research Questions

**RQ1 —** What are the representative attrition patterns of vehicles characterized as regular transportation versus enthusiast-oriented, based on vehicle specification attributes?\
**RQ2 —** Can annual vehicle attrition be predicted for a given vehicle descriptor, and can the model accurately predict attrition across multiple annual periods?

---

## Hypothesis

Vehicles that exhibit a deceleration in annual attrition rate are candidates for emerging collectability. As collectors begin preserving a vehicle, scrapping slows, registered population stabilizes, and the attrition rate declines - a behavioral shift that may appear in registration data before the broader market recognizes the vehicle's collector status.

This analysis investigates whether that signal is detectable and consistent, and whether it is reinforced by structural vehicle characteristics such as drivetrain, body style, engine displacement, and transmission type. Vehicles showing both behavioral deceleration and enthusiast-aligned specifications represent the strongest candidates for emerging collectability.

The hypothesis is investigative, not assumed. The data will determine whether the signal exists, how early it appears, and whether it differs meaningfully across vehicle types.

---

## Data

| Table | Rows | Description |
| --- | --- | --- |
| `VIO_COHORT_ENRICHED` | 362K | Primary analysis table — cohort-level registration counts by snapshot with VINtelligence specs joined |
| `VIN_MODEL_AGG` | 28K | VINtelligence specs aggregated to model grain (model year + make + model) |
| `VIO_MATCHED` | 1.9B | Cleaned and filtered VIO registration records |
| `MAKE_CROSSWALK` | 159 | VIO make name to VINtelligence make code, matched on VINtelligence's full make name |

**Key columns in `VIO_COHORT_ENRICHED`:**

| Column | Type | Description |
| --- | --- | --- |
| `YEAR_MODEL` | Integer | Vehicle model year |
| `MAKE` | String | Vehicle make |
| `MODEL` | String | Vehicle model |
| `VINT_MAKE_CD` | String | VINtelligence make code (join key) |
| `SNAPSHOT_DATE` | Date | Observation date — quarterly snapshots from April 2021 to July 2026 (July and October 2021 are not available) |
| `VIO_COUNT` | Integer | Registration count for this cohort at this snapshot |
| `BASE_MSRP` | Float | Lowest base MSRP across the model's VIN patterns (zero placeholders treated as missing) |
| `MAX_ENGINE_DISP` | Float | Largest engine displacement available (cubic inches) |
| `MAX_CYLINDERS` | Integer | Largest cylinder count available |
| `HAS_MANUAL` | Flag (0/1) | Manual transmission offered, as the primary or an optional transmission |
| `HAS_RWD` | Flag (0/1) | Rear-wheel drive offered (excludes rear-wheel drive with 4x4) |
| `PRIMARY_DRIVE_TYPE` | String | Most common drive type across VIN patterns |
| `PRIMARY_BODY_STYLE` | String | Most common body style across VIN patterns |
| `MAX_DOOR_CNT` | Integer | Maximum door count available |
| `PRIMARY_VEH_TYPE` | String | Most common vehicle type classification |
| `PRIMARY_FUEL_TYPE` | String | Most common fuel type |
| `SEGMENTATION` | String | Most common market segment classification |
| `MAX_SHIP_WEIGHT` | Float | Largest shipping weight in pounds (zero placeholders treated as missing) |
| `ORIGIN` | String | Most common vehicle origin |
| `CONFIG_COUNT` | Integer | Number of distinct configurations (trim, body style, displacement, drive type, transmission) offered for the model; a measure of variety |
| `VIO_VEHICLE_TYPE` | String | Vehicle type recorded in VIO (car or light truck); used to classify cohorts without VINtelligence specs |

"Most common" values are the mode across VIN patterns, not weighted by vehicles produced or registered.

**Known limitations:**

- 11.4% of cohort rows have NULL spec values, primarily pre-1981 vehicles and niche or low-volume manufacturers. These rows are retained for attrition curve visualization but excluded from clustering and predictive modeling, which require complete feature vectors.
- 1.3% of VIO records were excluded during integration due to no exact model match in VINtelligence, primarily motorcycles, ATVs, discontinued models, and commercial or specialty vehicles. Two VIO makes (Austin Rover, Capri) have no VINtelligence equivalent.
- VINtelligence provides no values for ISO performance indicator, advanced vehicle type, or EV range; these columns are dropped.
- Rotary engines and unknown cylinder counts are recorded as missing in `MAX_CYLINDERS`.
- Attrition is compared year over year across all vehicle types.
- A small number of cohorts carry implausible model years (e.g. a 1966 Acura) or represent non-US-market vehicles (Canadian and Mexican market models) with very low counts.
- The most recent model years are still being sold into the market, so their registration counts are growing rather than declining.

---

## Notebook Structure

1. **Environment Setup** — connect to Snowflake, import libraries
2. **Data Loading & Schema Review** — load `VIO_COHORT_ENRICHED`, confirm shape and types
3. **Exploratory Data Analysis** — distributions, snapshot coverage, null audit, cohort composition
4. **Feature Engineering** — compute attrition metrics: peak VIO, vehicle age, annual attrition rate, cumulative attrition, attrition acceleration
5. **Initial Observations** — attrition curve visualization, segment comparisons, preliminary hypotheses
6. **Data Quality & Preparation Notes** — document cleaning decisions and scope boundaries

## Environment Setup

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

# Snowflake
from snowflake.snowpark import Session

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.3f}'.format)
%matplotlib inline

# connection settings live in ~/.snowflake/connections.toml (outside the repo); sign in opens in the browser
session = Session.builder.config("connection_name", "vehicle_attrition").create()

## Data Loading & Schema Review

In [ ]:
df_cohort = session.table("WORKSPACE.ENGINEER.VIO_COHORT_ENRICHED").to_pandas()

df_cohort['SNAPSHOT_DATE'] = pd.to_datetime(df_cohort['SNAPSHOT_DATE'])
df_cohort['YEAR_MODEL'] = df_cohort['YEAR_MODEL'].astype(int)

# scope: cars and trucks only; motorcycle data retained upstream for future analysis
df_cohort = df_cohort[df_cohort['VIO_VEHICLE_TYPE'].isin(['CAR', 'LIGHT TRUCK'])]

df_cohort.info()

## Exploratory Data Analysis

**Distribution**

- VIO Count Distribution: Tells you the range of population sizes across cohorts, ensuring attrition rates are compared fairly regardless of how common or rare a vehicle is.
- Model Year Distribution: Tells you how many cohorts exist per decade and where your data coverage is densest.
- Spec Column Distribution: Tells you whether your enthusiast-indicator variables have enough information to be useful. If majority of vehicles have the same body style code, that variable will not help the clustering algorithm distinguish anything.

**Correlation Matrix**

- Uses `cohort_specs` with peak VIO from `cohort_summary` merged in to build a combined dataframe called `corr_data`
- `corr_data` is built only for the correlation matrix since it does not contain time series and pre-1981 cohorts - all of which are not enriched with upstream specification data

**Summary Statistics (Cars and Trucks, 16,188 cohorts) -**

- Model years range from 1966 to 2027 (median 2004); 1966 is the earliest model year in the VIO data.
- 14,339 cohorts (88.6%) have VINtelligence specifications; availability varies by field (e.g. MSRP 13,484, transmission 11,816).
- Among cohorts with known data, 41% offered a manual transmission and ~50% offered rear-wheel drive.
- Specification ranges are plausible: cylinders reach 16 (Bugatti W16), displacement reaches 636 ci, and MSRP reaches $5.4M (Bugatti Mistral).
- Configuration count is right-skewed (median 3 for cars and 5 for trucks; maximum 153).

In [ ]:
spec_cols = ['BASE_MSRP', 'MAX_ENGINE_DISP', 'MAX_CYLINDERS', 'HAS_MANUAL', 'HAS_RWD',
             'PRIMARY_DRIVE_TYPE', 'PRIMARY_BODY_STYLE', 'MAX_DOOR_CNT', 'PRIMARY_VEH_TYPE',
             'PRIMARY_FUEL_TYPE', 'SEGMENTATION', 'MAX_SHIP_WEIGHT', 'ORIGIN', 'CONFIG_COUNT']

# one row per cohort: specs are identical across a cohort's snapshots, so any row works;
# VIO_COUNT is dropped because it varies by snapshot
cohort_specs = (df_cohort
                .drop_duplicates(subset=['YEAR_MODEL', 'MAKE', 'MODEL'])
                [['YEAR_MODEL', 'MAKE', 'MODEL', 'VIO_VEHICLE_TYPE'] + spec_cols].copy())

# vehicle class separates distribution categories (car specs != truck specs)
# VINtelligence class first; VIO vehicle type as fallback for cohorts without specs (pre-1981)
vint_class = {'P': 'Car', 'T': 'Truck'}
vio_class = {'CAR': 'Car', 'LIGHT TRUCK': 'Truck'}

cohort_specs['VEHICLE_CLASS'] = (cohort_specs['PRIMARY_VEH_TYPE'].map(vint_class)
                                 .fillna(cohort_specs['VIO_VEHICLE_TYPE'].map(vio_class)))
cohort_specs['CLASS_SOURCE'] = np.where(cohort_specs['PRIMARY_VEH_TYPE'].notna(), 'VINtelligence', 'VIO')

# check: UNMAPPED row should be 0
# pd.crosstab(cohort_specs['VEHICLE_CLASS'].fillna('UNMAPPED'), cohort_specs['CLASS_SOURCE']).reset_index()

In [ ]:
# fixed class order and colors, reused in every chart so each class keeps the same color
class_order = ['Car', 'Truck']
class_colors = {'Car': '#2a78d6', 'Truck': '#eb6834'}

# one row per cohort with its peak VIO and vehicle class; validate raises if the merge duplicates a cohort
cohort_summary = (df_cohort.groupby(['YEAR_MODEL', 'MAKE', 'MODEL'])['VIO_COUNT'].max().reset_index()
                  .merge(cohort_specs[['YEAR_MODEL', 'MAKE', 'MODEL', 'VEHICLE_CLASS']],
                         on=['YEAR_MODEL', 'MAKE', 'MODEL'], how='left', validate='one_to_one'))

In [ ]:
edges = 10 ** np.histogram_bin_edges(np.log10(cohort_summary['VIO_COUNT']), bins='fd')

fig, axes = plt.subplots(1, len(class_order), figsize=(5 * len(class_order), 4), sharex=True)
for ax, cls in zip(axes, class_order):
    vals = cohort_summary.loc[cohort_summary['VEHICLE_CLASS'] == cls, 'VIO_COUNT']
    ax.hist(vals, bins=edges, color=class_colors[cls], edgecolor='white', linewidth=0.5)
    ax.set_xscale('log')
    ax.set_title(f'{cls} ({len(vals):,} cohorts, median {vals.median():,.0f})')
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('Number of Cohorts')
fig.supxlabel('Peak VIO Count per Cohort (log scale)')
fig.suptitle('Distribution of Peak VIO Count by Vehicle Class')
plt.tight_layout()
plt.show()

**Observations -**

- Trucks have larger cohorts than cars: median peak VIO is ~9.9k for trucks versus ~1.6k for cars, reflecting fewer truck models with high registration volume (e.g. full-size pickups and SUVs).
- Cars span a wider range, from single-digit exotics to cohorts above 100k, while most trucks cluster between 1k and 100k.

**Thoughts -**

- Attrition must be measured as a percentage, since cohort sizes span several orders of magnitude.
- Small cohorts produce unstable attrition rates (losing 1 of 5 vehicles is 20%), so a minimum size threshold or weighting, set per class, is needed before modeling.

In [ ]:
year_class = (cohort_summary.groupby(['YEAR_MODEL', 'VEHICLE_CLASS']).size()
              .unstack(fill_value=0)[class_order])

fig, ax = plt.subplots(figsize=(12, 5))
bottom = np.zeros(len(year_class))
for cls in class_order:
    ax.bar(year_class.index, year_class[cls], bottom=bottom, width=1.0,
           color=class_colors[cls], edgecolor='white', linewidth=0.5, label=cls)
    bottom += year_class[cls].values

ax.axvline(1980.5, color='#52514e', linestyle='--', linewidth=1)
ax.text(1981, ax.get_ylim()[1] * 0.95, 'VINtelligence specs begin (1981)',
        color='#52514e', fontsize=9, va='top')

ax.set_xlabel('Model Year')
ax.set_ylabel('Number of Cohorts (Make + Model)')
ax.set_title('Distribution of Cohorts by Model Year and Vehicle Class')
ax.legend(frameon=False, loc='upper left')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

**Observations -**

- Coverage begins at model year 1966. Pre-1981 cohorts (1,849) have registration history but no VINtelligence specifications.
- The number of car models peaks around 2017 and declines afterward, while truck models grow steadily; trucks outnumber cars from roughly 2022 onward, consistent with the industry shift toward pickups and SUVs.
- Model years 2026 and 2027 are incomplete because those vehicles are still being sold into the market.

**Thoughts -**

- Pre-1981 cohorts are key collector-era vehicles and are kept for attrition analysis, but excluded from modeling that requires specifications.
- Model years still being sold must be excluded from attrition until their registrations stop growing.
- As trucks and SUVs make up a growing share of models, future collector candidates may increasingly come from these segments, which supports benchmarking attrition within each class.

In [ ]:
msrp_all = cohort_specs['BASE_MSRP'].dropna()
edges = 10 ** np.histogram_bin_edges(np.log10(msrp_all), bins='fd')

fig, axes = plt.subplots(1, len(class_order), figsize=(5 * len(class_order), 4), sharex=True)
for ax, cls in zip(axes, class_order):
    cls_rows = cohort_specs[cohort_specs['VEHICLE_CLASS'] == cls]
    vals = cls_rows['BASE_MSRP'].dropna()
    missing = cls_rows['BASE_MSRP'].isna().mean()
    ax.hist(vals, bins=edges, color=class_colors[cls], edgecolor='white', linewidth=0.5)
    ax.set_xscale('log')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.set_title(f'{cls} ({len(vals):,} cohorts, median ${vals.median():,.0f})\n{missing:.1%} missing MSRP',
                 fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('Number of Cohorts')
fig.supxlabel('Base MSRP (log scale, nominal dollars)')
fig.suptitle('Distribution of Base MSRP by Vehicle Class')
plt.tight_layout()
plt.show()

# cohort_specs.nlargest(10, 'BASE_MSRP')[['YEAR_MODEL', 'MAKE', 'MODEL', 'VEHICLE_CLASS', 'BASE_MSRP']]
# cohort_specs.groupby('VEHICLE_CLASS')['BASE_MSRP'].describe(percentiles=[0.10, 0.50, 0.90, 0.99]).loc[class_order]

**Observations -**

- Cars and trucks share a similar median MSRP (~$26.6k), but cars have a much longer high-price tail (99th percentile ~$408k versus ~$128k for trucks), including a distinct exotic and luxury cluster around $200k to $500k.
- The highest-priced cars are all limited-production hypercars (e.g. Bugatti Chiron, Ferrari F60 America), confirming the tail reflects real vehicles.
- Missing MSRP (~17%) is mostly pre-1981 cohorts; among cohorts with specifications, ~5% of cars and ~7% of trucks lack an MSRP.

**Thoughts -**

- Higher relative price is a candidate enthusiast indicator.
- Hypercars have very small populations and are collectible by design, so emerging collectability is more likely to appear among mainstream vehicles.
- MSRP is in nominal dollars; a within-model-year price rank will be used to remove inflation.

In [ ]:
disp_all = cohort_specs['MAX_ENGINE_DISP'].dropna()
edges = np.histogram_bin_edges(disp_all, bins='fd')

fig, axes = plt.subplots(1, len(class_order), figsize=(5 * len(class_order), 4), sharex=True)
for ax, cls in zip(axes, class_order):
    cls_rows = cohort_specs[cohort_specs['VEHICLE_CLASS'] == cls]
    vals = cls_rows['MAX_ENGINE_DISP'].dropna()
    missing = cls_rows['MAX_ENGINE_DISP'].isna().mean()
    ax.hist(vals, bins=edges, color=class_colors[cls], edgecolor='white', linewidth=0.5)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.set_title(f'{cls} ({len(vals):,} cohorts, median {vals.median():,.0f} ci)\n{missing:.1%} missing displacement',
                 fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('Number of Cohorts')
fig.supxlabel('Max Engine Displacement (cubic inches)')
fig.suptitle('Distribution of Max Engine Displacement by Vehicle Class')
plt.tight_layout()
plt.show()

# cohort_specs.groupby('VEHICLE_CLASS')['MAX_ENGINE_DISP'].describe(percentiles=[0.10, 0.50, 0.90, 0.99]).loc[class_order]

**Observations -**

- Median max displacement is ~~183 ci (~~3.0L) for cars and ~~230 ci (~~3.8L) for trucks; trucks run larger at every percentile.
- Displacement clusters at common engine sizes rather than forming a smooth curve, reflecting shared engine families (four-cylinder, V6, V8) reused across many models. Trucks show a distinct big-block cluster near ~~455 ci (~~7.4L to 7.5L).
- The car maximum of 512 ci (8.4L) matches the Dodge Viper V10, supporting data validity.

**Thoughts -**

- Large displacement, particularly in cars, is a candidate performance and enthusiast indicator.
- Because displacement groups into engine families, it is likely strongly correlated with cylinder count; the correlation analysis will determine whether both are needed as model features.
- Electric vehicles have no displacement, so missing values must be treated as "no engine," not "unknown."

In [ ]:
# cylinder counts are discrete, so bars per value; shares make car and truck panels comparable
# 0 = electric; rotary and unknown cylinder codes are recorded as missing
cyl_values = sorted(cohort_specs['MAX_CYLINDERS'].dropna().unique())
cyl_labels = [f'{v:g}' for v in cyl_values]

fig, axes = plt.subplots(1, len(class_order), figsize=(5 * len(class_order), 4), sharey=True)
for ax, cls in zip(axes, class_order):
    cls_rows = cohort_specs[cohort_specs['VEHICLE_CLASS'] == cls]
    vals = cls_rows['MAX_CYLINDERS'].dropna()
    missing = cls_rows['MAX_CYLINDERS'].isna().mean()
    shares = vals.value_counts(normalize=True).reindex(cyl_values, fill_value=0) * 100
    ax.bar(cyl_labels, shares.values, color=class_colors[cls], edgecolor='white', linewidth=0.5)
    ax.set_title(f'{cls} ({len(vals):,} cohorts)\n{missing:.1%} missing cylinder count', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('Share of Cohorts (%)')
fig.supxlabel('Max Cylinder Count (0 = electric)')
fig.suptitle('Distribution of Max Cylinder Count by Vehicle Class')
plt.tight_layout()
plt.show()

# cohort counts per cylinder value, by class
# pd.crosstab(cohort_specs['MAX_CYLINDERS'], cohort_specs['VEHICLE_CLASS'])[class_order]

**Observations -**

- Cars are led by four-cylinder engines (~~35%), followed by V6 (~~31%) and V8 (~~24%); trucks skew larger, led by V8 (~~37%) and V6 (~36%).
- V10 engines are more common in trucks (~~1.5%) than cars (~~1%), consistent with heavy-duty pickup engines.
- ~5% of car cohorts offer a V12. This reflects two effects: the maximum available engine is recorded (e.g. a V12 top trim counts for the whole model), and low-volume exotic makers contribute many small cohorts.
- Electric vehicles are mostly recorded with a missing cylinder count rather than 0; fuel type identifies them more reliably (3.3% of cars, 3.8% of trucks).

**Thoughts -**

- High cylinder counts (V10, V12, W16) are strong performance indicators, but because they reflect the top available engine rather than the typical one, they identify models that offered high performance rather than vehicles that were typically high performance.
- Cohort shares describe the variety of models, not the vehicle population; weighting by registrations would show a much smaller V12 share.

In [ ]:
# manual share among cohorts with known transmission data, by model year
min_known = 30   # hide years where the share rests on too few cohorts
grp = cohort_specs.groupby(['YEAR_MODEL', 'VEHICLE_CLASS'])['HAS_MANUAL']
manual_by_year = (grp.mean() * 100).unstack()[class_order]
known_by_year = grp.count().unstack()[class_order]
manual_by_year = manual_by_year.where(known_by_year >= min_known).loc[1981:2025]

fig, ax = plt.subplots(figsize=(10, 4))
for cls in class_order:
    ax.plot(manual_by_year.index, manual_by_year[cls], color=class_colors[cls], linewidth=2, label=cls)
ax.set_ylim(0, 100)
ax.set_xlabel('Model Year')
ax.set_ylabel('Manual Offered (%)')
ax.set_title(f'Manual Transmission Availability by Model Year (years with {min_known}+ known cohorts)')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

**Observations -**

- Manual transmission availability declines steadily over time in both classes. Cars fall from ~60% of models through the late 1990s and 2000s to ~11% to 16% in the 2020s; trucks fall faster, to ~2% by 2025.
- Early 1980s values rest on few cohorts, because most 1981 to 1984 models have no transmission data in VINtelligence.

**Thoughts -**

- Because manual availability depends strongly on era, the flag alone overstates the enthusiast signal for older vehicles and understates it for recent ones. A manual offered in a 2020 model is far more distinctive than one in a 1990 model.
- Feature engineering should compare manual availability against the model's own era (e.g. relative to the share of models offering a manual in the same model year), the same approach planned for MSRP.

In [ ]:
# VINtelligence stores body style as a code; use the most common description for each code as its label
body_lookup = session.sql("""
    SELECT BODY_STYLE_CD AS CODE, MODE(BODY_STYLE_DESC) AS LABEL
    FROM WORKSPACE.ENGINEER.VINTELLIGENCE_RAW
    WHERE BODY_STYLE_CD IS NOT NULL
    GROUP BY 1
""").to_pandas()
body_labels = dict(zip(body_lookup['CODE'], body_lookup['LABEL'].str.title()))

cohort_specs['BODY_STYLE_NAME'] = cohort_specs['PRIMARY_BODY_STYLE'].map(body_labels)

# top 10 body styles per class; the rest grouped as Other
top_n = 10
fig, axes = plt.subplots(1, len(class_order), figsize=(6 * len(class_order), 4.5), sharex=True)
for ax, cls in zip(axes, class_order):
    cls_rows = cohort_specs[cohort_specs['VEHICLE_CLASS'] == cls]
    vals = cls_rows['BODY_STYLE_NAME'].dropna()
    missing = cls_rows['PRIMARY_BODY_STYLE'].isna().mean()
    shares = vals.value_counts(normalize=True) * 100
    if len(shares) > top_n:
        shares = pd.concat([shares.head(top_n), pd.Series({'Other': shares.iloc[top_n:].sum()})])
    ax.barh(shares.index[::-1], shares.values[::-1],
            color=class_colors[cls], edgecolor='white', linewidth=0.5)
    ax.set_xlabel('Share of Cohorts (%)')
    ax.set_title(f'{cls} ({len(vals):,} cohorts)\n{missing:.1%} missing body style', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('Primary Body Style by Vehicle Class (Top 10)')
plt.tight_layout()
plt.show()

# # cohort counts per body style, by class
# pd.crosstab(cohort_specs['BODY_STYLE_NAME'], cohort_specs['VEHICLE_CLASS'])[class_order] \
#   .sort_values(class_order[0], ascending=False)

**Observations -**

- Sedans make up half of car cohorts (4,250; 50.2%), followed by coupes (1,685; 19.9%), hatchbacks (1,304; 15.4%), convertibles (760; 9.0%), and wagons (436; 5.1%).
- Coupes and convertibles, the classic enthusiast body styles, together account for 2,445 car cohorts (28.9%).
- The truck class is dominated by SUVs (3,390; 57.8%), followed by pickups (980; 16.7%) and vans (921; 15.7%).
- 527 truck cohorts (9.0%) have commercial body styles (straight truck, cab chassis, cutaway, strip chassis, step van, motor home).
- 14 truck cohorts are convertibles, likely open-top SUVs such as the Jeep Wrangler.
- Body style is the most common style across a model's VIN patterns, so secondary styles such as a convertible version of a coupe are undercounted.

**Thoughts -**

- Coupe and convertible body styles are candidate enthusiast indicators for cars.
- Commercial body styles provide a way to separate commercial vehicles from light-duty trucks and SUVs, which is likely needed because commercial vehicles follow different usage and attrition patterns.
- If convertibles prove important, a separate "convertible offered" flag would capture models where the convertible was not the most common version.

In [ ]:
# VINtelligence stores market segment as a code; use the most common description for each code as its label
seg_lookup = session.sql("""
    SELECT SEGMENTATION_CD AS CODE, MODE(SEGMENTATION_DESC) AS LABEL
    FROM WORKSPACE.ENGINEER.VINTELLIGENCE_RAW
    WHERE SEGMENTATION_CD IS NOT NULL
    GROUP BY 1
""").to_pandas()
seg_labels = dict(zip(seg_lookup['CODE'], seg_lookup['LABEL']))

cohort_specs['SEGMENT_NAME'] = cohort_specs['SEGMENTATION'].map(seg_labels)

# top 10 segments per class; the rest grouped as Other
top_n = 10
fig, axes = plt.subplots(1, len(class_order), figsize=(6.5 * len(class_order), 4.5), sharex=True)
for ax, cls in zip(axes, class_order):
    cls_rows = cohort_specs[cohort_specs['VEHICLE_CLASS'] == cls]
    vals = cls_rows['SEGMENT_NAME'].dropna()
    missing = cls_rows['SEGMENTATION'].isna().mean()
    shares = vals.value_counts(normalize=True) * 100
    if len(shares) > top_n:
        shares = pd.concat([shares.head(top_n), pd.Series({'Other': shares.iloc[top_n:].sum()})])
    ax.barh(shares.index[::-1], shares.values[::-1],
            color=class_colors[cls], edgecolor='white', linewidth=0.5)
    ax.set_xlabel('Share of Cohorts (%)')
    ax.set_title(f'{cls} ({len(vals):,} cohorts)\n{missing:.1%} missing segment', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('Market Segment by Vehicle Class (Top 10)')
plt.tight_layout()
plt.show()

# # cohort counts per segment, by class (reset_index keeps segment names in exports)
# pd.crosstab(cohort_specs['SEGMENT_NAME'], cohort_specs['VEHICLE_CLASS'])[class_order] \
#   .sort_values(class_order[0], ascending=False).reset_index()

**Observations -**

- Car segments group into three families: mainstream (41.3%), luxury (34.7%), and sport (23.1%, across Luxury Sport, Compact Sport, Prestige Sport, and Mid Size Sport).
- Truck segments are dominated by utility (SUV) segments (57.8%, of which 21.7% are luxury utility), followed by pickups (18.8%), vans (17.0%), and the Commercial Truck segment (6.4%).
- Utility segment counts (3,341) closely match SUV body style counts (3,390), supporting consistency between the two fields.
- Missing segment is mostly pre-1981 cohorts; only ~2% to 3% of cohorts with specifications lack a segment.

**Thoughts -**

- Sport segments are the most direct enthusiast indicator available, since they are an explicit classification rather than a proxy such as price or displacement.
- Grouping segments into families (sport, luxury, mainstream for cars; utility, pickup, van, commercial for trucks) makes segment usable as a model feature and resolves truck fragmentation.
- The Commercial Truck segment, alongside commercial body styles, supports separating commercial vehicles from light-duty trucks and SUVs.

In [ ]:
# VINtelligence stores drive type as a code; use the most common description for each code as its label
drive_lookup = session.sql("""
    SELECT DRV_TYP_CD AS CODE, MODE(DRV_TYP_DESC) AS LABEL
    FROM WORKSPACE.ENGINEER.VINTELLIGENCE_RAW
    WHERE DRV_TYP_CD IS NOT NULL
    GROUP BY 1
""").to_pandas()
drive_labels = dict(zip(drive_lookup['CODE'], drive_lookup['LABEL']))

cohort_specs['DRIVE_TYPE_NAME'] = cohort_specs['PRIMARY_DRIVE_TYPE'].map(drive_labels)

# fixed order so each drive type sits in the same row in both panels
drive_order = [drive_labels[c] for c in ['RWD', '4RD', 'FWD', '4FD', 'AWD', 'U'] if c in drive_labels]

fig, axes = plt.subplots(1, len(class_order), figsize=(6 * len(class_order), 4), sharex=True)
for ax, cls in zip(axes, class_order):
    cls_rows = cohort_specs[cohort_specs['VEHICLE_CLASS'] == cls]
    vals = cls_rows['DRIVE_TYPE_NAME'].dropna()
    missing = cls_rows['PRIMARY_DRIVE_TYPE'].isna().mean()
    shares = vals.value_counts(normalize=True).reindex(drive_order, fill_value=0) * 100
    ax.barh(shares.index[::-1], shares.values[::-1],
            color=class_colors[cls], edgecolor='white', linewidth=0.5)
    ax.set_xlabel('Share of Cohorts (%)')
    ax.set_title(f'{cls} ({len(vals):,} cohorts)\n{missing:.1%} missing drive type', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('Primary Drive Type by Vehicle Class')
plt.tight_layout()
plt.show()

# # cohort counts per drive type, by class
# pd.crosstab(cohort_specs['DRIVE_TYPE_NAME'], cohort_specs['VEHICLE_CLASS'])[class_order] \
#   .reindex(drive_order).reset_index()

**Observations -**

- Front-wheel drive is the most common primary drive type for cars (3,849; 45.5%), followed by rear-wheel drive (3,307; 39.1%) and all-wheel drive (1,311; 15.5%). No car cohorts are primarily 4x4.
- Trucks split across rear-wheel drive (1,871; 31.9%), all-wheel drive (1,829; 31.2%), rear-wheel drive with 4x4 (1,084; 18.5%), and front-wheel drive (1,001; 17.1%), with front-wheel drive with 4x4 rare (83; 1.4%).
- Half of truck cohorts (2,955; 50.4%) are rear-drive based (rear-wheel drive or rear-wheel drive with 4x4), reflecting pickups and body-on-frame SUVs; the remainder are largely front- and all-wheel-drive crossovers and vans.
- Primary rear-wheel drive shares are lower than the "rear-wheel drive offered" flag (cars 39.1% vs 49%, trucks 31.9% vs 50%), because many models were offered in more than one drive type.
- Drive type is complete for all cohorts with specifications; missing values match the pre-1981 share exactly, and no cohort is primarily "Unknown."

**Thoughts -**

- Rear-wheel drive is a meaningful enthusiast indicator for cars because it is not the default: front-wheel drive is more common.
- The primary drive type and the rear-wheel drive flag answer different questions (typical configuration versus availability), so both may be useful as features.
- Like manual transmissions, rear-wheel drive availability in cars likely shifted over time as front-wheel drive became standard; interpreting it relative to model year may strengthen the signal.

In [ ]:
# VINtelligence stores fuel type as a code; use the most common description for each code as its label
fuel_lookup = session.sql("""
    SELECT ENG_FUEL_CD AS CODE, MODE(ENG_FUEL_DESC) AS LABEL
    FROM WORKSPACE.ENGINEER.VINTELLIGENCE_RAW
    WHERE ENG_FUEL_CD IS NOT NULL
    GROUP BY 1
""").to_pandas()
fuel_labels = dict(zip(fuel_lookup['CODE'], fuel_lookup['LABEL']))

cohort_specs['FUEL_TYPE_NAME'] = cohort_specs['PRIMARY_FUEL_TYPE'].map(fuel_labels)

# fixed order (most common overall first) so each fuel sits in the same row in both panels
fuel_order = cohort_specs['FUEL_TYPE_NAME'].value_counts().index.tolist()

fig, axes = plt.subplots(1, len(class_order), figsize=(6 * len(class_order), 4), sharex=True)
for ax, cls in zip(axes, class_order):
    cls_rows = cohort_specs[cohort_specs['VEHICLE_CLASS'] == cls]
    vals = cls_rows['FUEL_TYPE_NAME'].dropna()
    missing = cls_rows['PRIMARY_FUEL_TYPE'].isna().mean()
    shares = vals.value_counts(normalize=True).reindex(fuel_order, fill_value=0) * 100
    bars = ax.barh(shares.index[::-1], shares.values[::-1],
                   color=class_colors[cls], edgecolor='white', linewidth=0.5)
    # labels make small fuels readable next to the dominant gasoline bar
    ax.bar_label(bars, labels=[f'{v:.1f}%' for v in shares.values[::-1]],
                 fontsize=8, color='#52514e', padding=2)
    ax.set_xlabel('Share of Cohorts (%)')
    ax.set_title(f'{cls} ({len(vals):,} cohorts)\n{missing:.1%} missing fuel type', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('Primary Fuel Type by Vehicle Class')
plt.tight_layout()
plt.show()

# cohort counts per fuel type, by class
# pd.crosstab(cohort_specs['FUEL_TYPE_NAME'], cohort_specs['VEHICLE_CLASS'])[class_order] \
#   .reindex(fuel_order).reset_index()

**Observations -**

- Gasoline dominates both classes (cars 91.5%, trucks 84.2%).
- Diesel is far more common in trucks (5.9%) than cars (0.5%), consistent with heavy-duty pickups and commercial vehicles; flex-fuel is also more common in trucks (4.2% vs 1.0%).
- Electric vehicles account for 3.3% of car cohorts and 3.8% of truck cohorts; hybrids account for 3.3% and 1.7%.
- The number of electric cohorts closely matches the number of post-1981 cohorts missing engine displacement, indicating that missing displacement for these vehicles reflects having no engine rather than missing data.
- Fuel type is complete for all cohorts with specifications; missing values match the pre-1981 share exactly.
- Primary fuel type is the most common fuel across a model's VIN patterns, so hybrid and flex-fuel versions of mainly gasoline models are undercounted.

**Thoughts -**

- Electric vehicles need explicit handling in engine-based features: missing displacement and cylinder counts for electric vehicles should be treated as "no engine," and fuel type can be used to identify them.
- Electric and hybrid cohorts are concentrated in recent model years, so they will have limited attrition history within the data window.

In [ ]:
# VINtelligence stores origin as a code; use the most common description for each code as its label
# (MODE also resolves inconsistent casing in the source, e.g. 'DOMESTIC' vs 'Domestic')
origin_lookup = session.sql("""
    SELECT ORGN_CD AS CODE, MODE(ORGN_DESC) AS LABEL
    FROM WORKSPACE.ENGINEER.VINTELLIGENCE_RAW
    WHERE ORGN_CD IS NOT NULL
    GROUP BY 1
""").to_pandas()
origin_labels = dict(zip(origin_lookup['CODE'], origin_lookup['LABEL']))

cohort_specs['ORIGIN_NAME'] = cohort_specs['ORIGIN'].map(origin_labels)

# logical order from domestic to import, so both panels read the same way
origin_order = [origin_labels[c] for c in ['D', 'Z', 'B', 'I'] if c in origin_labels]

fig, axes = plt.subplots(1, len(class_order), figsize=(6 * len(class_order), 3.5), sharex=True)
for ax, cls in zip(axes, class_order):
    cls_rows = cohort_specs[cohort_specs['VEHICLE_CLASS'] == cls]
    vals = cls_rows['ORIGIN_NAME'].dropna()
    missing = cls_rows['ORIGIN'].isna().mean()
    shares = vals.value_counts(normalize=True).reindex(origin_order, fill_value=0) * 100
    ax.barh(shares.index[::-1], shares.values[::-1],
            color=class_colors[cls], edgecolor='white', linewidth=0.5)
    ax.set_xlabel('Share of Cohorts (%)')
    ax.set_title(f'{cls} ({len(vals):,} cohorts)\n{missing:.1%} missing origin', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('Vehicle Origin by Vehicle Class')
plt.tight_layout()
plt.show()

# # cohort counts per origin, by class
# pd.crosstab(cohort_specs['ORIGIN_NAME'], cohort_specs['VEHICLE_CLASS'])[class_order] \
#   .reindex(origin_order).reset_index()

**Observations -**

- Most car cohorts are imports (5,057; 60.4%), with domestic cars at 28.3%; trucks lean domestic (2,784; 47.8%), with imports at 31.0%.
- Foreign-brand vehicles built in North America are twice as common among trucks (17.8%) as cars (8.8%), reflecting the North American production of foreign-brand SUVs and pickups.
- US brands account for 30.7% of car cohorts but 51.2% of truck cohorts, consistent with US brands' long-standing strength in pickups and full-size SUVs.
- Shares count models, not vehicles; foreign brands offer many low-volume models, so imports are over-represented relative to registered vehicles.
- Origin is missing for ~1% of cohorts with specifications (99 cars, 48 trucks) beyond the pre-1981 cohorts.

**Thoughts -**

- Origin groups very different vehicles under "Import" (e.g. Japanese, German, Italian), so a country-of-origin feature derived from make would be more useful for collectability analysis, such as testing collector interest in 1990s Japanese vehicles.
- Weighting by registrations would give a more realistic picture of the vehicle population than cohort counts.

In [ ]:
weight_all = cohort_specs['MAX_SHIP_WEIGHT'].dropna()
edges = np.histogram_bin_edges(weight_all, bins='fd')

fig, axes = plt.subplots(1, len(class_order), figsize=(5 * len(class_order), 4), sharex=True)
for ax, cls in zip(axes, class_order):
    cls_rows = cohort_specs[cohort_specs['VEHICLE_CLASS'] == cls]
    vals = cls_rows['MAX_SHIP_WEIGHT'].dropna()
    missing = cls_rows['MAX_SHIP_WEIGHT'].isna().mean()
    ax.hist(vals, bins=edges, color=class_colors[cls], edgecolor='white', linewidth=0.5)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.set_title(f'{cls} ({len(vals):,} cohorts, median {vals.median():,.0f} lbs)\n{missing:.1%} missing weight',
                 fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('Number of Cohorts')
fig.supxlabel('Max Shipping Weight (lbs)')
fig.suptitle('Distribution of Max Shipping Weight by Vehicle Class')
plt.tight_layout()
plt.show()

# cohort_specs.nsmallest(5, 'MAX_SHIP_WEIGHT')[['YEAR_MODEL', 'MAKE', 'MODEL', 'VEHICLE_CLASS', 'MAX_SHIP_WEIGHT']]
# cohort_specs[cohort_specs['VEHICLE_CLASS'] == 'Car'].nlargest(5, 'MAX_SHIP_WEIGHT')[['YEAR_MODEL', 'MAKE', 'MODEL', 'MAX_SHIP_WEIGHT']]
# cohort_specs.groupby('VEHICLE_CLASS')['MAX_SHIP_WEIGHT'].describe(percentiles=[0.10, 0.50, 0.90, 0.99]).loc[class_order]

**Observations -**

- Median shipping weight is 3,450 lbs for cars and 4,456 lbs for trucks; 80% of cars fall between ~2,400 and 4,400 lbs, and 80% of trucks between ~3,300 and 6,100 lbs.
- Cars form a single peak around 3,500 lbs, while trucks show a second cluster around 5,500 to 6,000 lbs (heavy-duty pickups, full-size vans, and large SUVs) and a thin commercial tail up to ~9,700 lbs.
- Extremes are plausible except one: the lightest cohorts are low-speed electric vehicles (~1,140 lbs) and the heaviest trucks are heavy-duty pickups and vans, but the heaviest car (2011 Lexus LS, 7,420 lbs) is a source data error from a single VIN pattern.
- Weight is the least complete specification: beyond pre-1981 cohorts, it is missing for 570 cars (6.7%) and 598 trucks (10.2%) with specifications.

**Thoughts -**

- Displacement relative to weight is a simple power-to-weight proxy that may separate performance vehicles from ordinary ones better than either measure alone.
- Maximum-based specifications reflect the heaviest or most capable version of a model (e.g. heavy-duty Silverado) and are sensitive to a single erroneous VIN pattern, so outliers should be checked before modeling.

In [ ]:
# door counts are discrete, so bars per value; shares make car and truck panels comparable
door_values = sorted(cohort_specs['MAX_DOOR_CNT'].dropna().unique())
door_labels = [f'{v:g}' for v in door_values]

fig, axes = plt.subplots(1, len(class_order), figsize=(5 * len(class_order), 4), sharey=True)
for ax, cls in zip(axes, class_order):
    cls_rows = cohort_specs[cohort_specs['VEHICLE_CLASS'] == cls]
    vals = cls_rows['MAX_DOOR_CNT'].dropna()
    missing = cls_rows['MAX_DOOR_CNT'].isna().mean()
    shares = vals.value_counts(normalize=True).reindex(door_values, fill_value=0) * 100
    ax.bar(door_labels, shares.values, color=class_colors[cls], edgecolor='white', linewidth=0.5)
    ax.set_title(f'{cls} ({len(vals):,} cohorts)\n{missing:.1%} missing door count', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('Share of Cohorts (%)')
fig.supxlabel('Max Door Count')
fig.suptitle('Distribution of Max Door Count by Vehicle Class')
plt.tight_layout()
plt.show()

# cohort counts per door count, by class
# pd.crosstab(cohort_specs['MAX_DOOR_CNT'], cohort_specs['VEHICLE_CLASS'])[class_order].reset_index()

**Observations -**

- Most cohorts have a maximum of 4 doors (cars 68.3%, trucks 74.8%); 2-door cohorts make up 30.9% of cars and 15.1% of trucks.
- 3-door configurations appear mainly in trucks (9.7%), likely extended-cab pickups and minivans with a single sliding door; cars rarely have odd door counts, so hatchbacks are recorded by passenger doors.
- The 2-door car share (30.9%) closely matches the coupe and convertible body style share (28.9%), showing that door count largely duplicates body style.
- Door count is complete for cars with specifications; 226 truck cohorts (3.9%) with specifications lack it.

**Thoughts -**

- Door count adds little beyond body style and is a candidate for removal during feature selection; the correlation analysis should confirm this.

In [ ]:
# number of distinct VIN patterns per cohort: how many configurations (trim, engine, body, drive) were offered
pattern_all = cohort_specs['CONFIG_COUNT'].dropna()
edges = 10 ** np.histogram_bin_edges(np.log10(pattern_all), bins='fd')

fig, axes = plt.subplots(1, len(class_order), figsize=(5 * len(class_order), 4), sharex=True)
for ax, cls in zip(axes, class_order):
    cls_rows = cohort_specs[cohort_specs['VEHICLE_CLASS'] == cls]
    vals = cls_rows['CONFIG_COUNT'].dropna()
    missing = cls_rows['CONFIG_COUNT'].isna().mean()
    ax.hist(vals, bins=edges, color=class_colors[cls], edgecolor='white', linewidth=0.5)
    ax.set_xscale('log')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.set_title(f'{cls} ({len(vals):,} cohorts, median {vals.median():,.0f} configurations)\n{missing:.1%} missing',
                 fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('Number of Cohorts')
fig.supxlabel('Configurations per Cohort (log scale)')
fig.suptitle('Distribution of Configuration Count by Vehicle Class')
plt.tight_layout()
plt.show()

# cohort_specs.groupby('VEHICLE_CLASS')['CONFIG_COUNT'].describe(percentiles=[0.10, 0.50, 0.90, 0.99]).loc[class_order]

**Observations -**

- Trucks are offered in more configurations than cars: median configuration count is 5 for trucks versus 3 for cars, and the 99th percentile is 59 versus 24.
- About a quarter of car cohorts have a single configuration, indicating models offered in only one trim, engine, and body combination.
- Configuration count replaced VIN pattern count, which was distorted by manufacturers encoding different levels of detail in their VINs (e.g. 2015 Volvo V60: 1,032 VIN patterns but 26 configurations) and by some models storing individual vehicle VINs.
- Configuration count is complete for all cohorts with specifications.

**Thoughts -**

- Configuration count measures variety, not production volume; it may act as a proxy for production scale if it correlates with peak VIO, which the correlation analysis will test.
- The right skew suggests a log transformation before using it as a model feature.

In [ ]:
# 0/1 indicators for categories tied to the hypotheses; NaN stays NaN so a missing value is never counted as "no"
def indicator(col, values):
    return cohort_specs[col].isin(values).astype(float).where(cohort_specs[col].notna())

sport_segments = ['Luxury Sport', 'Compact Sport', 'Prestige Sport', 'Mid Size Sport']
luxury_segments = ['Mid Size Luxury Car', 'Compact Luxury Car', 'Full Size Luxury Car', 'Subcompact Luxury Car',
                   'Full Size Luxury Prestige', 'Compact Luxury Utility', 'Subcompact Plus Luxury Utility',
                   'Lower Mid Size Luxury Utility', 'Upper Mid Size Luxury Utility', 'Full Size Luxury Utility',
                   'Full Size Luxury Prestige Utility', 'Subcompact Luxury Utility', 'Full Size Half Ton Luxury Pickup']
commercial_styles = ['Straight Truck', 'Cab Chassis', 'Cutaway', 'Incomplete (Strip Chassis)', 'Step Van', 'Motor Home']

cohort_specs['IS_SPORT_SEGMENT']  = indicator('SEGMENT_NAME', sport_segments)
cohort_specs['IS_LUXURY_SEGMENT'] = indicator('SEGMENT_NAME', luxury_segments)
cohort_specs['IS_COUPE_CONV']     = indicator('BODY_STYLE_NAME', ['Coupe', 'Convertible'])
cohort_specs['IS_COMMERCIAL']     = indicator('BODY_STYLE_NAME', commercial_styles)
cohort_specs['IS_FWD']            = indicator('DRIVE_TYPE_NAME', ['Front Wheel Drive'])
cohort_specs['IS_ELECTRIC']       = indicator('FUEL_TYPE_NAME', ['Electric'])
cohort_specs['IS_IMPORT']         = indicator('ORIGIN_NAME', ['Import', 'Import Built in North America'])

# sanity check: share of cohorts flagged, by class (NaN = source value missing)
cohort_specs.groupby('VEHICLE_CLASS')[[c for c in cohort_specs if c.startswith('IS_')]].mean().round(3).reindex(class_order)

In [ ]:
# Spearman rank correlation: suited to skewed, discrete, and 0/1 features, and robust to outliers
corr_cols = {'BASE_MSRP': 'MSRP', 'MAX_ENGINE_DISP': 'Displacement', 'MAX_CYLINDERS': 'Cylinders',
             'MAX_SHIP_WEIGHT': 'Weight', 'MAX_DOOR_CNT': 'Doors', 'HAS_MANUAL': 'Manual',
             'HAS_RWD': 'RWD', 'CONFIG_COUNT': 'Configurations', 'PEAK_VIO': 'Peak VIO',
             'YEAR_MODEL': 'Model Year',
             'IS_SPORT_SEGMENT': 'Sport Segment', 'IS_LUXURY_SEGMENT': 'Luxury Segment',
             'IS_COUPE_CONV': 'Coupe/Convertible', 'IS_COMMERCIAL': 'Commercial',
             'IS_FWD': 'Front-Wheel Drive', 'IS_ELECTRIC': 'Electric', 'IS_IMPORT': 'Import Brand'}

# one row per cohort with specs, plus each cohort's peak VIO
corr_data = (cohort_specs[cohort_specs['CONFIG_COUNT'].notna()]
             .merge(cohort_summary[['YEAR_MODEL', 'MAKE', 'MODEL', 'VIO_COUNT']],
                    on=['YEAR_MODEL', 'MAKE', 'MODEL'], how='left', validate='one_to_one')
             .rename(columns={'VIO_COUNT': 'PEAK_VIO'}))

# diverging scale: blue = negative, neutral gray = none, red = positive
div_cmap = LinearSegmentedColormap.from_list('div', ['#2a78d6', '#f0efec', '#e34948'])

top_pairs = {}
fig, axes = plt.subplots(1, len(class_order), figsize=(9 * len(class_order), 8))
for ax, cls in zip(axes, class_order):
    cls_data = corr_data.loc[corr_data['VEHICLE_CLASS'] == cls, list(corr_cols)].rename(columns=corr_cols)
    cls_data = cls_data.loc[:, cls_data.nunique() > 1]   # drop indicators with no variation in this class
    corr = cls_data.corr(method='spearman')
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)   # show each pair once
    sns.heatmap(corr, mask=mask, ax=ax, cmap=div_cmap, vmin=-1, vmax=1, center=0,
                annot=True, fmt='.2f', annot_kws={'size': 7}, square=True,
                linewidths=0.5, linecolor='white', cbar=(cls == class_order[-1]))
    ax.set_title(f'{cls} ({len(cls_data):,} cohorts)')

    # strongest pairs for the written observations
    lower = corr.where(np.tril(np.ones_like(corr, dtype=bool), k=-1)).stack()
    top_pairs[cls] = lower.reindex(lower.abs().sort_values(ascending=False).index).head(12).round(2)

fig.suptitle('Spearman Correlation of Vehicle Features by Class')
plt.tight_layout()
plt.show()

# 12 strongest correlations per class (positive or negative)
# pd.concat(top_pairs, axis=1)

**Observations -**

- Displacement and cylinder count are nearly redundant (0.94 cars, 0.91 trucks) and both correlate with weight (0.61 to 0.71), forming a single engine size and weight cluster.
- Door count largely duplicates body style and segment in cars (−0.79 with coupe/convertible, −0.76 with sport segment); in trucks it tracks model year (0.58) and MSRP (0.55) instead.
- Sport segment is only weakly correlated with MSRP, displacement, cylinders, manual, and RWD (0.15 to 0.27), so it adds information beyond price and engine size. It overlaps partly with coupe/convertible (0.66).
- Manual transmission and rear-wheel drive are essentially uncorrelated in cars (−0.08), so they are distinct signals.
- Model year is strongly related to MSRP (0.60 cars, 0.76 trucks) and negatively to manual availability (−0.35 cars, −0.54 trucks). Adjusting for model year only partly reduces the manual-MSRP correlation (cars −0.49 to −0.40, trucks −0.66 to −0.45), so manual availability remains associated with lower-priced vehicles within the same model year.
- Configuration count is moderately related to peak VIO (0.40 cars, 0.54 trucks), a partial proxy for production scale.

**Thoughts -**

- For modeling, keep one of displacement or cylinder count, and drop door count for cars.
- Keep sport segment, manual, and RWD as separate features; sport segment and coupe/convertible partly overlap.
- The era effect supports the planned within-model-year MSRP rank and era-relative manual availability, but manual availability is not a clean enthusiast signal, since it also marks lower-priced vehicles. Its value may depend on combination with other features (e.g. sport segment, RWD).
- Peak VIO is partly a survivorship measure for older cohorts, so it should not stand in for production volume without caution.